In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import mean_squared_error

In [2]:
# Load datasets
tournament_df = pd.read_csv("Dataset/Tournament Matchups.csv")
kenpom_df = pd.read_csv("Dataset/KenPom Barttorvik.csv")

# Basic inspection
print("Tournament Matchups shape:", tournament_df.shape)
print("KenPom Barttorvik shape:", kenpom_df.shape)

tournament_df.head()

Tournament Matchups shape: (2140, 8)
KenPom Barttorvik shape: (1147, 103)


,YEAR,BY YEAR NO,TEAM NO,TEAM,SEED,ROUND,CURRENT ROUND,SCORE
0,2025,2140,1141,Auburn,1,4,64,83
1,2025,2139,1145,Alabama St.,16,64,64,63
2,2025,2138,1119,Louisville,8,64,64,75
3,2025,2137,1134,Creighton,9,32,64,89
4,2025,2136,1114,Michigan,5,16,64,68


In [3]:
kenpom_df.head()

,YEAR,CONF,CONF ID,QUAD NO,QUAD ID,TEAM NO,TEAM ID,TEAM,SEED,ROUND,...,BADJT RANK,AVG HGT RANK,EFF HGT RANK,EXP RANK,TALENT RANK,FT% RANK,OP FT% RANK,PPPO RANK,PPPD RANK,ELITE SOS RANK
0,2025,MAC,17,66,2,1147,2,Akron,13,64,...,19,361,364,73,149,86,239,25,145,305
1,2025,SEC,28,66,2,1146,3,Alabama,2,8,...,1,57,4,261,32,211,92,9,174,1
2,2025,SWAC,31,68,4,1145,4,Alabama St.,16,64,...,154,285,239,83,251,270,61,227,147,253
3,2025,Pat,25,66,2,1144,6,American,16,68,...,349,303,248,181,364,58,34,218,163,329
4,2025,B12,7,66,2,1143,8,Arizona,4,16,...,52,74,129,225,16,18,143,25,87,3


In [4]:
# missing values in tournament_df
print(tournament_df.isnull().sum())

# missing values in kenpom_df
print(kenpom_df.isnull().sum())

YEAR             0
BY YEAR NO       0
TEAM NO          0
TEAM             0
SEED             0
ROUND            0
CURRENT ROUND    0
SCORE            0
dtype: int64
YEAR              0
CONF              0
CONF ID           0
QUAD NO           0
QUAD ID           0
                 ..
FT% RANK          0
OP FT% RANK       0
PPPO RANK         0
PPPD RANK         0
ELITE SOS RANK    0
Length: 103, dtype: int64


In [5]:
# distribution of scores
tournament_df["SCORE"].describe()

count    2140.000000
mean       69.821963
std        12.003840
min        29.000000
25%        61.000000
50%        70.000000
75%        78.000000
max       113.000000
Name: SCORE, dtype: float64

In [6]:
# tournament years
print(tournament_df["YEAR"].unique())
# kenpom years
print(kenpom_df["YEAR"].unique())

[2025 2024 2023 2022 2021 2019 2018 2017 2016 2015 2014 2013 2012 2011
 2010 2009 2008]
[2025 2024 2023 2022 2021 2019 2018 2017 2016 2015 2014 2013 2012 2011
 2010 2009 2008]


In [7]:
# team names in each dataset
print(tournament_df["TEAM"].unique()[:20])

print(kenpom_df["TEAM"].unique()[:20])

['Auburn' 'Alabama St.' 'Louisville' 'Creighton' 'Michigan' 'UC San Diego'
 'Texas A&M' 'Yale' 'Mississippi' 'North Carolina' 'Iowa St.' 'Lipscomb'
 'Marquette' 'New Mexico' 'Michigan St.' 'Bryant' 'Florida' 'Norfolk St.'
 'Connecticut' 'Oklahoma']
['Akron' 'Alabama' 'Alabama St.' 'American' 'Arizona' 'Arkansas' 'Auburn'
 'Baylor' 'Bryant' 'BYU' 'Clemson' 'Colorado St.' 'Connecticut'
 'Creighton' 'Drake' 'Duke' 'Florida' 'Georgia' 'Gonzaga' 'Grand Canyon']


In [8]:
# merge on year and team
merged_df = pd.merge(
    tournament_df,
    kenpom_df,
    on=["YEAR", "TEAM"],
    how="inner"
)

print(merged_df.shape)

merged_df.head()

(2131, 109)


,YEAR,BY YEAR NO,TEAM NO_x,TEAM,SEED_x,ROUND_x,CURRENT ROUND,SCORE,CONF,CONF ID,...,BADJT RANK,AVG HGT RANK,EFF HGT RANK,EXP RANK,TALENT RANK,FT% RANK,OP FT% RANK,PPPO RANK,PPPD RANK,ELITE SOS RANK
0,2025,2140,1141,Auburn,1,4,64,83,SEC,28,...,143,22,66,20,90,116,264,3,59,2
1,2025,2139,1145,Alabama St.,16,64,64,63,SWAC,31,...,154,285,239,83,251,270,61,227,147,253
2,2025,2138,1119,Louisville,8,64,64,75,ACC,2,...,92,65,58,23,129,104,143,43,35,59
3,2025,2137,1134,Creighton,9,32,64,89,BE,8,...,187,7,3,219,30,140,143,84,81,52
4,2025,2136,1114,Michigan,5,16,64,68,B10,6,...,65,5,2,116,38,162,239,94,47,22


In [9]:
# initial feature columns
features = ["SEED_x", "BADJ O", "BADJ D", "BARTHAG", "EXP", "TALENT"]

X = merged_df[features]
# for now score, will change to margin of victory later
y = merged_df["SCORE"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape)
print(X_test.shape)

(1704, 6)
(427, 6)


In [10]:
# normalize
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [11]:
# mlp model
model = keras.Sequential([ 
    layers.Dense(64, activation="relu", input_shape=(X_train_scaled.shape[1],)), 
    layers.Dense(32, activation="relu"), 
    layers.Dense(16, activation="relu"), 
    layers.Dropout(0.2), 
    layers.Dense(8, activation="relu"), 
    layers.Dense(1) 
])

/opt/anaconda3/lib/python3.12/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [12]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,201 (12.50 KB)

 Trainable params: 3,201 (12.50 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
history = model.fit(
    X_train_scaled,
    y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

Epoch 1/100
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 4958.1528 - mae: 69.4199 - val_loss: 4837.0420 - val_mae: 68.3779
Epoch 2/100
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 904us/step - loss: 4449.4126 - mae: 65.5610 - val_loss: 2804.5811 - val_mae: 50.7033
Epoch 3/100
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 769us/step - loss: 1766.2336 - mae: 38.2158 - val_loss: 646.0477 - val_mae: 21.1825
Epoch 4/100
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 768us/step - loss: 733.0489 - mae: 22.4414 - val_loss: 412.0796 - val_mae: 16.6212
Epoch 5/100
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 745us/step - loss: 503.2830 - mae: 18.3578 - val_loss: 293.8941 - val_mae: 13.9285
Epoch 6/100
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 772us/step - loss: 364.4919 - mae: 14.9368 - val_loss: 231.6570 - val_mae: 12.3194
Epoch 7/100
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 778us/step - loss: 344.3739 - mae: 14.8337 - val_loss: 201.2748 - val_mae: 11.4308
Epoch 8/100
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 749us/step - loss: 294.0401 - mae: 13.3534 - val_loss: 197.3919 - val_mae: 11.29

In [14]:
test_loss, test_mae = model.evaluate(X_test_scaled, y_test)

print("Test MSE:", test_loss)
print("Test MAE:", test_mae)

14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 740us/step - loss: 284.8843 - mae: 14.5091
Test MSE: 299.2200927734375
Test MAE: 14.792155265808105


In [16]:
predictions = model.predict(X_test_scaled)
print(predictions[:10])

14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 858us/step
[[61.464012]
 [52.375225]
 [50.81779 ]
 [47.086613]
 [54.890625]
 [53.576004]
 [53.51561 ]
 [55.49841 ]
 [53.7211  ]
 [56.828484]]
